In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Locate project files
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_path = project_root / "data" / "hotel_bookings_clean.csv"

# Load cleaned dataset
df = pd.read_csv(data_path)

# Define target and identifier
target = df["is_canceled"]
row_ids = df["row_id"]

# Stratified 80/20 split
train_ids, test_ids = train_test_split(
    row_ids,
    test_size=0.20,
    random_state=42,
    stratify=target
)

# Check that the split is valid
assert len(set(train_ids) & set(test_ids)) == 0
assert len(train_ids) + len(test_ids) == len(df)

print("Total records:", len(df))
print("Training records:", len(train_ids))
print("Test records:", len(test_ids))

print("\nOverall cancellation rate:", round(target.mean(), 4))

train_df = df[df["row_id"].isin(train_ids)]
test_df = df[df["row_id"].isin(test_ids)]

print("Training cancellation rate:", round(train_df["is_canceled"].mean(), 4))
print("Test cancellation rate:", round(test_df["is_canceled"].mean(), 4))

In [ ]:

from pathlib import Path
import pandas as pd

# Define the output folder
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
split_dir = project_root / "data" / "splits"
split_dir.mkdir(parents=True, exist_ok=True)

# Save the exact row IDs used in the random split
train_path = split_dir / "train_ids.csv"
test_path = split_dir / "test_ids.csv"

pd.DataFrame({
    "row_id": train_ids
}).to_csv(train_path, index=False)

pd.DataFrame({
    "row_id": test_ids
}).to_csv(test_path, index=False)

# Reload the saved files for verification
saved_train = pd.read_csv(train_path)
saved_test = pd.read_csv(test_path)

# Check row counts
assert len(saved_train) == 94851
assert len(saved_test) == 23713

# Check uniqueness and no overlap
assert saved_train["row_id"].is_unique
assert saved_test["row_id"].is_unique

assert set(saved_train["row_id"]).isdisjoint(
    set(saved_test["row_id"])
)

# Check all original records are included exactly once
assert (
    set(saved_train["row_id"]) |
    set(saved_test["row_id"])
) == set(df["row_id"])

print("Training IDs saved:", len(saved_train))
print("Test IDs saved:", len(saved_test))
print("No overlapping IDs: True")
print("All records accounted for: True")
print("Files saved successfully.")
print("Folder:", split_dir)
